# P2 — Perp-specific + Cross-asset + Vol-regime (BTC/ETH perp, 1h sampling)

**Slug**: `crypto_intraday/20_p2_strategies`

**Status**: in_progress

**Research questions**:
1. Can the best P1 gross signal (`volume_shock_continuation`, Sharpe 1.11 zero-cost) survive realistic costs at 1h sampling (~12× less turnover than 5m)?
2. Does the perp-specific information (funding rate) carry tradable signal that close-only doesn't?
3. Do cross-asset signals (BTC ↔ ETH spread / beta residual) survive costs?
4. Does a vol-regime overlay (only trade when realized vol is moderate) improve the best survivor?

**Method**: same window as P1 (Q1+Q2 2024) but RESAMPLED to 1h. P1's horizon-discovery decision picked 1h as the working interval; this notebook honors that choice. Per-symbol slip + funding for `perp_base` / `perp_stress`.

**Decision rule**: `accept_monitoring` requires net > 0 AND Sharpe > 0.5 AT STRESS costs.

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.holdout import PMHoldout, access_summary_for_report
from alpha_lab.data.loaders import binance_vision as bv
from alpha_lab.features import intraday as f
from alpha_lab.utils.config import load_config

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 200)

In [ ]:
cfg = load_config('crypto_intraday')
holdout = PMHoldout.from_config('crypto_intraday')

SYMBOLS = ['BTCUSDT', 'ETHUSDT']
BARS_PER_YEAR_1H = 365 * 24   # 8,760
RES_START, RES_END = '2024-01-01', '2024-07-01'
PER_SYMBOL_CAP = 0.5

panel_1m = bv.load_klines(
    SYMBOLS, '1m', start=RES_START, end=RES_END, market='perp', holdout=holdout,
)

def resample_ohlcv(df: pd.DataFrame, offset: str) -> pd.DataFrame:
    return df.resample(offset, label='right', closed='right').agg({
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
        'volume': 'sum', 'quote_volume': 'sum', 'n_trades': 'sum',
        'taker_buy_base': 'sum',
    }).dropna(how='all')

ohlcv_1h = {s: resample_ohlcv(df, '1h') for s, df in panel_1m.frames.items()}
prices_1h = pd.DataFrame({s: o['close'] for s, o in ohlcv_1h.items()})
common_idx = prices_1h.dropna().index
prices_1h = prices_1h.loc[common_idx]
ohlcv_1h = {s: o.loc[common_idx] for s, o in ohlcv_1h.items()}

funding = bv.load_funding(SYMBOLS, start=RES_START, end=RES_END, holdout=holdout)
print(f'1h bars: {len(common_idx):,}  funding events: {len(funding)}')

In [ ]:
@dataclass(frozen=True)
class CostScenario:
    name: str
    fee_bps: float
    slip_bps: dict[str, float]
    use_funding: bool

scenarios = []
for key in ['zero', 'perp_base', 'perp_stress']:
    block = cfg['costs'][key]
    scenarios.append(CostScenario(
        name=key,
        fee_bps=float(block['fee_bps_per_side']),
        slip_bps={s: float(block['slippage_bps_per_side'][s]) for s in SYMBOLS},
        use_funding=bool(block.get('include_funding', False)),
    ))

## Strategies

All features computed at 1h. Most strategies use threshold-style entry/exit so signals persist naturally.

In [ ]:
def _zsig(s: pd.Series, w: int = 48) -> pd.Series:
    return s / s.rolling(w, min_periods=12).std()

# --- Turnover-controlled P1 leaders, now at 1h ---------------------------
def volume_shock_continuation_v2() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        o = ohlcv_1h[sym]
        vz = f.volume_zscore(o['volume'], window=24)   # 24h volume z-score
        r = f.log_return(o['close'], window=1)
        sig = pd.Series(0.0, index=vz.index)
        shock = vz > 2.0
        sig[shock & (r > 0)] = 1.0
        sig[shock & (r < 0)] = -1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def bollinger_mr_threshold() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        bb = f.bollinger_pct_b(ohlcv_1h[sym]['close'], window=20)
        sig = pd.Series(0.0, index=bb.index)
        sig[bb < 0.1] = 1.0
        sig[bb > 0.9] = -1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def rsi_mr_threshold() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        rsi = f.rsi(ohlcv_1h[sym]['close'], window=14)
        sig = pd.Series(0.0, index=rsi.index)
        sig[rsi < 30] = 1.0
        sig[rsi > 70] = -1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

# --- Perp-specific (funding) ---------------------------------------------
funding_z_events = f.funding_zscore(funding, window=30)
funding_cum_events = f.funding_cumulative(funding, window=10)
# Forward-fill onto the 1h grid (leak-safe: last-known funding at each bar)
funding_z_1h = funding_z_events.reindex(common_idx, method='ffill')
funding_cum_1h = funding_cum_events.reindex(common_idx, method='ffill')

def funding_contrarian() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        z = funding_z_1h[sym].fillna(0.0)
        sig = pd.Series(0.0, index=z.index)
        sig[z > 1.0] = -1.0
        sig[z < -1.0] = 1.0
        w[sym] = sig * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

def funding_momentum() -> pd.DataFrame:
    w = {}
    for sym in SYMBOLS:
        cum = funding_cum_1h[sym].fillna(0.0)
        w[sym] = np.sign(cum) * PER_SYMBOL_CAP
    return pd.DataFrame(w).fillna(0.0)

# --- Cross-asset ---------------------------------------------------------
def spread_z_pair_trade() -> pd.DataFrame:
    z = f.spread_zscore(prices_1h['BTCUSDT'], prices_1h['ETHUSDT'], window=48)
    sig = -z.clip(-2.0, 2.0) / 2.0
    return pd.DataFrame({
        'BTCUSDT': sig * PER_SYMBOL_CAP,
        'ETHUSDT': -sig * PER_SYMBOL_CAP,
    }).fillna(0.0)

def btc_leads_eth() -> pd.DataFrame:
    btc_ret_1h = f.log_return(prices_1h['BTCUSDT'], window=1)
    z = btc_ret_1h / btc_ret_1h.rolling(48, min_periods=12).std()
    eth_w = z.clip(-2.0, 2.0) / 2.0 * PER_SYMBOL_CAP
    return pd.DataFrame({
        'BTCUSDT': pd.Series(0.0, index=common_idx),
        'ETHUSDT': eth_w,
    }).fillna(0.0)

def beta_residual_mr() -> pd.DataFrame:
    br = f.rolling_beta_residual(prices_1h['ETHUSDT'], prices_1h['BTCUSDT'], window=48)
    z = br['residual'] / br['residual'].rolling(48, min_periods=12).std()
    eth_w = -z.clip(-2.0, 2.0) / 2.0 * PER_SYMBOL_CAP
    return pd.DataFrame({
        'BTCUSDT': pd.Series(0.0, index=common_idx),
        'ETHUSDT': eth_w,
    }).fillna(0.0)

STRATEGIES = {
    'volume_shock_continuation_v2': volume_shock_continuation_v2,
    'bollinger_mr_threshold':       bollinger_mr_threshold,
    'rsi_mr_threshold':             rsi_mr_threshold,
    'funding_contrarian':           funding_contrarian,
    'funding_momentum':             funding_momentum,
    'spread_z_pair_trade':          spread_z_pair_trade,
    'btc_leads_eth':                btc_leads_eth,
    'beta_residual_mr':             beta_residual_mr,
}

for name, fn in STRATEGIES.items():
    w = fn().reindex(common_idx).fillna(0.0)
    nz = int((w.abs().sum(axis=1) > 0).sum())
    print(f'{name:32s}: {nz:>5d} active bars of {len(common_idx)} ({100*nz/len(common_idx):.1f}%)')

## Master run

In [ ]:
rows = []
for strat_name, strat_fn in STRATEGIES.items():
    weights = strat_fn().reindex(common_idx).fillna(0.0)
    for sc in scenarios:
        res = run_backtest(
            weights, prices_1h,
            costs_bps=sc.fee_bps,
            slippage_bps=sc.slip_bps,
            funding=funding if sc.use_funding else None,
            bars_per_year=BARS_PER_YEAR_1H,
        )
        stats = summary(res.returns, periods=BARS_PER_YEAR_1H)
        rows.append({
            'strategy': strat_name,
            'scenario': sc.name,
            'gross_total': float((1 + res.gross_returns).prod() - 1),
            'net_total':   float((1 + res.returns).prod() - 1),
            'cost_drag':   float(res.costs.sum()),
            'funding_drag': float(res.funding_costs.sum()),
            'turnover_sum': float(res.turnover.sum()),
            'Sharpe':      stats.get('Sharpe', float('nan')),
            'MaxDD':       stats.get('MaxDD', float('nan')),
        })
results = pd.DataFrame(rows)
results

In [ ]:
net_pivot = results.pivot(index='strategy', columns='scenario', values='net_total')[['zero', 'perp_base', 'perp_stress']]
net_pivot = net_pivot.reindex(index=list(STRATEGIES.keys()))
print('Net total return — Q1+Q2 2024, 1h sampling:')
print(net_pivot)

In [ ]:
sharpe_pivot = results.pivot(index='strategy', columns='scenario', values='Sharpe')[['zero', 'perp_base', 'perp_stress']]
sharpe_pivot = sharpe_pivot.reindex(index=list(STRATEGIES.keys()))
print('Annualized Sharpe:')
print(sharpe_pivot)

In [ ]:
attrib = results[results.scenario == 'perp_base'][
    ['strategy', 'gross_total', 'cost_drag', 'funding_drag', 'net_total', 'turnover_sum']
].set_index('strategy')
print('Attribution under perp_base:')
print(attrib)

## Decisions per strategy

In [ ]:
decisions = []
for strat in STRATEGIES:
    zero  = results[(results.strategy == strat) & (results.scenario == 'zero')].iloc[0]
    base  = results[(results.strategy == strat) & (results.scenario == 'perp_base')].iloc[0]
    stress= results[(results.strategy == strat) & (results.scenario == 'perp_stress')].iloc[0]
    net_stress = float(stress['net_total'])
    sh_stress = float(stress['Sharpe']) if not pd.isna(stress['Sharpe']) else 0.0
    net_zero = float(zero['net_total'])
    sh_zero = float(zero['Sharpe']) if not pd.isna(zero['Sharpe']) else 0.0
    if net_stress > 0 and sh_stress > 0.5:
        decision = 'accept_monitoring'
    elif float(base['net_total']) > 0:
        decision = 'needs_revision'
    elif net_zero > 0 and sh_zero > 0.2:
        decision = 'reject_cost_killed'
    else:
        decision = 'reject'
    decisions.append({
        'strategy': strat, 'net_zero': net_zero, 'sh_zero': sh_zero,
        'net_base': float(base['net_total']),
        'net_stress': net_stress, 'sh_stress': sh_stress,
        'decision': decision,
    })
dec_df = pd.DataFrame(decisions)
print(dec_df.to_string(index=False))

## Vol-regime overlay on best stress survivor

In [ ]:
stress_subset = results[results.scenario == 'perp_stress'].set_index('strategy')
best = stress_subset['Sharpe'].fillna(-99).idxmax()
print(f'Best perp_stress Sharpe: {best} = {stress_subset.loc[best, "Sharpe"]:.4f}')

# Vol gate: only trade when realized vol is in bottom 60% of rolling distribution.
rvol_btc = f.realized_vol_close(prices_1h['BTCUSDT'], window=48)
rvol_eth = f.realized_vol_close(prices_1h['ETHUSDT'], window=48)
med_rvol = pd.concat([rvol_btc, rvol_eth], axis=1).mean(axis=1)
rvol_pctile = med_rvol.rolling(48 * 7, min_periods=48).rank(pct=True)
low_vol_gate = (rvol_pctile <= 0.6).astype(float)

base_w = STRATEGIES[best]().reindex(common_idx).fillna(0.0)
gated_w = base_w.mul(low_vol_gate.fillna(0.0), axis=0)

overlay_rows = []
for sc in scenarios:
    res = run_backtest(
        gated_w, prices_1h,
        costs_bps=sc.fee_bps, slippage_bps=sc.slip_bps,
        funding=funding if sc.use_funding else None,
        bars_per_year=BARS_PER_YEAR_1H,
    )
    stats = summary(res.returns, periods=BARS_PER_YEAR_1H)
    overlay_rows.append({
        'scenario': sc.name,
        'net_total': float((1+res.returns).prod() - 1),
        'Sharpe':    stats.get('Sharpe', float('nan')),
        'turnover':  float(res.turnover.sum()),
    })
overlay_df = pd.DataFrame(overlay_rows).set_index('scenario')
print(f'\n{best} + low-vol gate:')
print(overlay_df)
print(f'\nvs ungated {best}:')
print(results[results.strategy == best].set_index('scenario')[['net_total', 'Sharpe', 'turnover_sum']])

## PM holdout audit

In [ ]:
audit = access_summary_for_report()
for k, v in audit.items():
    print(f'  {k}: {v}')
if audit['accessed']:
    raise RuntimeError('PM HOLDOUT WAS ACCESSED — contaminated.')
print('\nPM Holdout was not accessed.')